<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Simple Linear Regression

*Session 5 · Notebook 02.02 · Lecture · Student version*

## Overview

Regression predicts a **continuous** number (unlike classification, which predicts a category). Simple linear regression is the starting point: it models the relationship between one feature and the target as a straight line. This notebook fits a line, uses it to predict, visualises the fit, evaluates it, and, crucially for risk work, **interprets** what the coefficients mean.

We use a small salary dataset (years of experience vs salary), which makes the mechanics and the interpretation easy to see.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain what linear regression fits and how (the line of best fit / least squares).
- Train a `LinearRegression` model and use it to predict.
- Visualise the fitted line on training and test data.
- Evaluate a regression with R-squared, MAE, MSE and RMSE.
- Interpret the slope and intercept in business terms.

## Prerequisites

- Session 5 notebook 01.02 (the scikit-learn workflow) and Session 4 (correlation).
- Comfort with `train_test_split`.

## Index

1. [Why this matters for risk analysis](#sec1)
2. [What is linear regression?](#sec2)
3. [Load and explore the data](#sec3)
4. [Split and train the model](#sec4)
5. [Predict and visualise the fit](#sec5)
6. [Evaluate the model](#sec6)
7. [Interpret the model](#sec7)
8. [Exercises](#exercises)
9. [Challenge](#challenge)
10. [Key Takeaways](#takeaways)
11. [Further Reading](#reading)

<a id="setup"></a>
# Section 0: Setup

We use `Salary_Data.csv` (30 rows: `YearsExperience` and `Salary`), read from the repo-root `datasets/` folder (two levels up).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Or read directly from the public S3 bucket (no local file needed):
# dataset = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/Salary_Data.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# dataset = pd.read_csv(session_datasets_http["Salary_Data"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# dataset = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/Salary_Data.csv", header=True, inferSchema=True).toPandas()
dataset = pd.read_csv('../../datasets/Session_5/Salary_Data.csv')
print('shape:', dataset.shape)
dataset.head()

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Classification predicts a yes/no (will this loan default?); **regression predicts a number**, and risk work is full of numbers to predict.

| Regression predicts a number | Risk example |
|---|---|
| Loss given default (LGD) | How much of the exposure is lost when a loan defaults |
| Exposure at default (EAD) | The balance expected at the time of default |
| Claim severity | The size of an insurance claim |
| Expected loss | Combining probability and amount into a monetary figure |

Linear regression is transparent: each coefficient states how much the target moves per unit of a driver, which is exactly the kind of explanation a risk model needs to be signed off.

<a id="sec2"></a>
# Section 2: What is linear regression?

**Definition:** simple linear regression models the target as a **straight line** of one feature: `y = b0 + b1 * x`. It chooses the line that minimises the total **squared error** between the line and the actual points (the 'least squares' fit).

**Example:** predicting salary from years of experience: each extra year adds a fixed amount to the predicted salary.

**Analogy:** drawing the single straight line that best threads through a cloud of points, as close to as many of them as possible.

**Explanation:**

- **`b1` (slope):** how much the target changes per one-unit increase in the feature.
- **`b0` (intercept):** the predicted target when the feature is zero.
- It assumes the relationship is roughly **linear**; if it curves, a polynomial or transform may be needed (later notebooks).
- We judge the fit with R-squared (share of variance explained) and error metrics (MAE, MSE, RMSE).

**The formula (how the line is found):**

Least squares chooses the intercept $b_0$ and slope $b_1$ that make the total squared error as small as possible, $\sum_i \left(y_i - (b_0 + b_1 x_i)\right)^2$. For one feature this has an exact, direct solution (no iteration, no searching):

$$b_1 = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}, \qquad b_0 = \bar{y} - b_1 \bar{x}$$

where $\bar{x}$ and $\bar{y}$ are the means of the feature and the target. You plug in the numbers and the coefficients come straight out. For how this generalises to many features (and to Ridge and Lasso), see the side-note notebook `05_18`.

**scikit-learn documentation:** [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

<a id="sec3"></a>
# Section 3: Load and explore the data

A quick scatter shows the relationship we are about to model: salary rises fairly linearly with experience, which is exactly what linear regression is built for.

<a id="sec4"></a>
# Section 4: Split and train the model

We separate the feature `X` from the target `y`, split into training and test sets, then fit a `LinearRegression`. (`X` is kept as a 2D column, which scikit-learn expects for features.)

<a id="sec5"></a>
# Section 5: Predict and visualise the fit

We predict the test salaries and plot the fitted line over both the training and the test points. The blue line is the model; the red points are the real data.

<a id="sec6"></a>
# Section 6: Evaluate the model

Regression is judged by how far its predictions fall from the truth:

- **MAE** (mean absolute error): the average error, in the target's units (easy to read).
- **MSE** / **RMSE**: penalise large errors more; RMSE is back in the target's units.
- **R-squared**: the share of the variance in the target the model explains (1.0 is perfect, 0 is no better than always predicting the mean).

<a id="sec7"></a>
# Section 7: Interpret the model

The whole appeal of linear regression is that you can read it. The fitted line is `Salary = intercept + slope * YearsExperience`. The **slope** is the extra salary per year of experience; the **intercept** is the model's baseline salary at zero years.

<a id="exercises"></a>
# Section 8: Exercises

### Exercise 1: Predict a new value

Use the fitted `regressor` to predict the salary for someone with **6 years** of experience. (Pass a 2D input, e.g. `[[6]]`.)

In [ ]:
# Your turn. Write your solution here:


### Exercise 2: Training vs test R-squared

Compute the R-squared on the **training** set and on the **test** set. Are they similar (a sign the model generalises)?

In [ ]:
# Your turn. Write your solution here:


### Exercise 3: Report the error in plain terms

Compute the MAE on the test set and write one sentence explaining what it means for this salary model.

In [ ]:
# Your turn. Write your solution here:


<a id="challenge"></a>
## Challenge (optional): check the residuals

A good linear model should leave **residuals** (actual minus predicted) that scatter randomly around zero with no pattern. Compute the residuals on the test set, plot them against the predicted values, and comment on whether a straight line looks like a reasonable fit here.

In [ ]:
# Your turn. Write your solution here:


### How to read a residual plot

The plot shows the residuals (actual minus predicted) against the predicted values, with a dashed line at zero:

- **Random cloud around 0, no shape** - a straight line is appropriate (what we see here).
- **A curve (for example a U-shape)** - the relationship is non-linear; try a polynomial or a transform.
- **A funnel (spread grows across the range)** - the error variance is not constant; often fixed by transforming the target.
- **A slope or systematic trend** - the model is biased across the range of predictions.
- **A point far from the rest** - a possible outlier worth investigating.

In short, good residuals look like featureless noise around zero; any leftover pattern means the straight line missed something.

<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| Linear regression | Fits a straight line `y = b0 + b1*x` by least squares |
| `LinearRegression().fit(X, y)` | Learn the slope and intercept |
| `.predict(X)` | Predict the target for new feature values |
| `.coef_`, `.intercept_` | The slope(s) and intercept, for interpretation |
| `r2_score` | Share of variance explained (1.0 perfect) |
| `mean_absolute_error` / RMSE | Typical prediction error, in the target's units |
| residual plot | Diagnostic: residuals should scatter randomly around 0 |


## Conclusion

You can now fit, use, evaluate and interpret a simple linear regression. The next notebook extends this to **multiple** features, and later notebooks add polynomial terms and regularisation (Ridge and Lasso).

<a id="reading"></a>
## Further Reading & Resources

- [scikit-learn: Linear Models](https://scikit-learn.org/stable/modules/linear_model.html) LinearRegression and beyond.
- [scikit-learn: regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics) R2, MAE, MSE, RMSE.
- [Interpreting linear regression](https://en.wikipedia.org/wiki/Simple_linear_regression) slope and intercept.